# 00 - Build Spreadsheet Template Notebook
## Instagram Competition Monitoring & Curation System

Notebook ini digunakan untuk **menginisialisasi dan membangun template Google Sheets** (11 sheet) secara aman langsung dari Google Colab.

### Fitur Keamanan `build_template`:
- **Append-Safe / Non-destructive**: Tidak pernah menghapus sheet yang sudah ada atau menimpa data admin.
- **11 Sheet Lengkap**: `Competition Report`, `System Log`, `Execution Log`, `Config`, `Run State`, `Source Accounts`, `Master Top University`, `Master Top Organizer`, `Grade Rules`, `Eligibility Rules`, `Search Keywords`.
- **Grid 2000 Baris**: Dibuat dengan 2000 baris agar penulisan rentang eksplisit admin/sistem selalu muat.
- **Data Validation & Formatting**: Dropdown Status Pemeriksaan di kolom O, format teks pada kolom Post ID di System Log.
- **Protected Ranges**: Kolom A (Competition ID) dan Kolom D (Grade Lomba) diproteksi (warning-only).
- **Grade ARRAYFORMULA Otomatis**: Memasang formula dinamis berbasis data sheet `Grade Rules` dan named ranges `TOP_UNIV_PATTERN` serta `TOP_ORG_PATTERN`.

### 0. Environment Check & Smart Install (wajib dijalankan pertama)
Sel pertama memastikan repositori ter-clone di Colab, lalu memeriksa Python, versi paket, import, dan signature API.
Sel kedua hanya memasang paket yang belum ada atau di bawah versi minimum (tidak merusak environment bawaan Colab).

In [ ]:
# @title 0a. Environment Check & Clone GitHub (jika di Colab)
import os, sys, subprocess
from pathlib import Path

REPO_URL = "https://github.com/fajrialfarizy/SPIKE-Competition-Searching.git"
REPO_NAME = "SPIKE-Competition-Searching"

# Jika dijalankan di Google Colab (/content) dan compmon belum ada, clone dari GitHub
if not (Path.cwd() / 'compmon').exists() and not (Path.cwd().parent / 'compmon').exists():
    if not (Path.cwd() / REPO_NAME).exists():
        print(f"Mengunduh repositori dari GitHub: {REPO_URL}...")
        subprocess.run(["git", "clone", REPO_URL], check=True)
    if (Path.cwd() / REPO_NAME).exists():
        os.chdir(Path.cwd() / REPO_NAME)
        print(f"Beralih ke direktori proyek: {os.getcwd()}")

_root = Path.cwd() if (Path.cwd() / 'compmon').exists() else Path.cwd().parent
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

from compmon.env_check import run_environment_check

run_environment_check()


In [ ]:
# @title 0b. Smart Install
from compmon.env_check import ensure_environment

ensure_environment()


### 1. Autentikasi Google Account & Kredensial Spreadsheet
Autentikasi menggunakan akun Google resmi pemilik spreadsheet via `google.auth.default()` + `gspread.authorize()`.

ID spreadsheet diambil secara otomatis dari Colab Secrets (`SPREADSHEET_ID`) jika sudah diatur, atau Anda dapat mengisinya langsung pada kolom parameter di bawah.

In [ ]:
# @title 1. Autentikasi & Masukkan ID Spreadsheet
from google.colab import auth, userdata
import google.auth
import gspread

# Autentikasi akun Google resmi
auth.authenticate_user()
creds, _ = google.auth.default()
gc = gspread.authorize(creds)

# Ambil ID dari Colab Secrets atau input manual
manual_spreadsheet_id = "" # @param {type:"string"}

try:
    secret_id = userdata.get('SPREADSHEET_ID')
except Exception:
    secret_id = None

SPREADSHEET_ID = manual_spreadsheet_id.strip() or secret_id

if not SPREADSHEET_ID:
    raise ValueError("SPREADSHEET_ID belum diisi! Masukkan ID spreadsheet pada form atau simpan di Colab Secrets.")

print(f"Target Spreadsheet ID: {SPREADSHEET_ID}")


### 2. Eksekusi Pembuatan / Inisialisasi Template
Menjalankan `populate_template` dari modul `templates/build_template.py`.

Formula (`Competition Report!D2` dan sel bantu `Grade Rules!G2:G3`) ditulis mengikuti **locale spreadsheet**: pemisah argumen `;` untuk locale Indonesia, `,` untuk locale AS/Inggris. Spreadsheet locale tidak diubah. Bila sebuah sel tetap menampilkan `#ERROR!`, kosongkan `G2:G3` lalu jalankan ulang sel ini.

In [ ]:
# @title 2. Jalankan Inisialisasi Template
import logging

# Fix 9: tampilkan pesan build_template (locale, zona waktu, formula) di Colab.
logging.basicConfig(level=logging.INFO, force=True)
logging.getLogger("compmon").setLevel(logging.INFO)
logging.getLogger("templates").setLevel(logging.INFO)

from templates.build_template import populate_template

print("Memulai pembuatan/pemeriksaan template 11 sheet...")
populate_template(gc, SPREADSHEET_ID)
print("
Inisialisasi template Google Sheets selesai dengan sukses!")

# Verifikasi daftar worksheet yang ada
sh = gc.open_by_key(SPREADSHEET_ID)
titles = [ws.title for ws in sh.worksheets()]
print(f"Total Sheet ({len(titles)}): {', '.join(titles)}")
